# ECON2041 week 8 lecture: more than one X

One row per Victorian local government area (LGA), 79 of them, for 2024. Same data as the week 6 lecture, picked up where that lecture stopped.

## Data sources

- Gaming expenditure: Victorian Gambling and Casino Control Commission, 2024
- Employment: Department of Employment, Small Area Labour Markets, 2024
- Population: Department of Environment, Land, Water and Planning, 2023
- Disadvantage: ABS SEIFA, 2021

## Setup

Our usual setup block, with one new import at the end: `summary_col`, which puts several regressions side by side in one table.

In [ ]:
# Our standard ECON2041 setup block
import numpy as np               # numerical tools (nicknamed np)
import pandas as pd              # data tools (nicknamed pd)
import matplotlib.pyplot as plt  # plotting tools (nicknamed plt)
import seaborn as sns            # statistical charts (nicknamed sns)
from statsmodels.formula.api import ols  # ordinary least squares regression
from statsmodels.iolib.summary2 import summary_col  # several models in one table

# Keep scalar output plain under NumPy 2 (0.5, not np.float64(0.5)).
if np.lib.NumpyVersion(np.__version__) >= "2.0.0":
    np.set_printoptions(legacy="1.25")

# Plot styling, set once here so every figure below inherits it
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["figure.figsize"] = (10, 6)

DATA = "https://emiliatjernstrom.com/econ2041/data"   # Unit datasets live at this web address

print("Setup done!")

## Load the data

We'll use these variables from the dataset:

| Variable | What it stores |
|---|---|
| `exp_per_adult` | Gambling expenditure (losses) per adult (\$/year) |
| `egm_per_1000` | Electronic gaming machines (EGMs, or pokies) per 1,000 adults, which we call pokie density |
| `ue_rate` | LGA unemployment rate, as a proportion |
| `SEIFA_dis_score` | Socio-Economic Indexes for Areas (SEIFA) disadvantage score, higher = less disadvantaged |

In [ ]:
# Load pokies-victoria-2024.csv into a dataframe called pokies
pokies = pd.read_csv(f"{DATA}/pokies-victoria-2024.csv")

pokies[["LGA", "exp_per_adult", "egm_per_1000", "ue_rate", "SEIFA_dis_score"]].head()

## Where we stopped: the simple regression

Losses per adult on pokie density, all 79 LGAs. `.summary().tables[1]` prints the coefficient table from the slides.

In [ ]:
# Fit losses per adult on pokie density with ordinary least squares
simple = ols("exp_per_adult ~ egm_per_1000", data=pokies).fit()

print(simple.summary().tables[1])

## Unemployment, holding pokie density fixed

### Building a dummy variable

`>` asks a yes-or-no question of every LGA at once, and `.astype(int)` turns the answers into 1 and 0. A 0/1 variable like this is called a dummy variable.

In [ ]:
# high_ue stores the comparison's answer, as 1 (above median) or 0 (below)
median_val = pokies["ue_rate"].median()
pokies["high_ue"] = (pokies["ue_rate"] > median_val).astype(int)

# A readable label for plotting, and one color per group so every figure matches
pokies["ue_group"] = np.where(pokies["high_ue"] == 1, "High unemployment", "Low unemployment")
UE_ORDER = ["Low unemployment", "High unemployment"]
UE_COLORS = {"Low unemployment": "#5b8fc9", "High unemployment": "#c8102e"}

pokies["ue_group"].value_counts()

### One regression per group

Same model, fit twice, on the two halves of the data. Each line only uses its own half.

In [ ]:
low_group = pokies[pokies["high_ue"] == 0]
high_group = pokies[pokies["high_ue"] == 1]

model_low = ols("exp_per_adult ~ egm_per_1000", data=low_group).fit()
model_high = ols("exp_per_adult ~ egm_per_1000", data=high_group).fit()

print("low unemployment, n =", int(model_low.nobs))
print(model_low.params.round(1))
print()
print("high unemployment, n =", int(model_high.nobs))
print(model_high.params.round(1))

In [ ]:
# sns.lmplot is regplot plus a hue argument: one scatterplot and one fitted line per group
sns.lmplot(data=pokies, x="egm_per_1000", y="exp_per_adult", hue="ue_group",
           hue_order=UE_ORDER, palette=UE_COLORS, ci=None,
           scatter_kws={"s": 40, "alpha": 0.7}, height=6, aspect=1.67)
plt.xlabel("EGMs per 1,000 adults")
plt.ylabel("Gambling losses per adult ($/year)")
plt.show()

## Multiple regression: both variables, one formula

A second explanatory variable goes to the right of `~`, separated by a `+`. The dummy shifts the intercept; the slope on pokie density is shared by both groups.

In [ ]:
dummy = ols("exp_per_adult ~ egm_per_1000 + high_ue", data=pokies).fit()

print(dummy.summary().tables[1])

### Two parallel lines

The estimate on `high_ue` is the vertical distance between the two lines: same slope, two intercepts. We build each line from the estimates by hand.

In [ ]:
# Pull the three estimates out by name
b0 = dummy.params["Intercept"]
b1 = dummy.params["egm_per_1000"]
b2 = dummy.params["high_ue"]

# 100 evenly spaced pokie-density values to draw the lines over
egm_grid = np.linspace(0, pokies["egm_per_1000"].max(), 100)

sns.scatterplot(data=pokies, x="egm_per_1000", y="exp_per_adult", hue="ue_group",
                hue_order=UE_ORDER, palette=UE_COLORS, s=40, alpha=0.7)
plt.plot(egm_grid, b0 + b1 * egm_grid, color=UE_COLORS["Low unemployment"])          # high_ue = 0
plt.plot(egm_grid, b0 + b2 + b1 * egm_grid, color=UE_COLORS["High unemployment"])    # high_ue = 1
plt.xlabel("EGMs per 1,000 adults")
plt.ylabel("Gambling losses per adult ($/year)")
plt.show()

### The second X can be continuous

Multiplying `ue_rate` by 100 gives a coefficient we can read per percentage point of unemployment.

In [ ]:
# ue_pct stores the unemployment rate in percentage points
pokies["ue_pct"] = 100 * pokies["ue_rate"]

percent = ols("exp_per_adult ~ egm_per_1000 + ue_pct", data=pokies).fit()

print(percent.summary().tables[1])

## Interactions: letting the slopes differ

A colon between two variables multiplies them. The product is the interaction term, and its coefficient is the difference in slopes between the two groups.

$$\text{losses}_i = \beta_0 + \beta_1 \, \text{EGMs}_i + \beta_2 \, \text{high\_ue}_i + \beta_3 \, (\text{EGMs}_i \times \text{high\_ue}_i) + u_i$$

In [ ]:
inter = ols("exp_per_adult ~ egm_per_1000 + high_ue + egm_per_1000:high_ue", data=pokies).fit()

print(inter.summary().tables[1])

### The two lines the interaction model implies

Low unemployment: intercept $\beta_0$, slope $\beta_1$. High unemployment: intercept $\beta_0 + \beta_2$, slope $\beta_1 + \beta_3$. Compare them with the split-sample lines above.

In [ ]:
b = inter.params

print("low unemployment:  intercept", round(b["Intercept"], 1),
      " slope", round(b["egm_per_1000"], 1))
print("high unemployment: intercept", round(b["Intercept"] + b["high_ue"], 1),
      " slope", round(b["egm_per_1000"] + b["egm_per_1000:high_ue"], 1))

In [ ]:
sns.scatterplot(data=pokies, x="egm_per_1000", y="exp_per_adult", hue="ue_group",
                hue_order=UE_ORDER, palette=UE_COLORS, s=40, alpha=0.7)
plt.plot(egm_grid, b["Intercept"] + b["egm_per_1000"] * egm_grid,
         color=UE_COLORS["Low unemployment"])
plt.plot(egm_grid, (b["Intercept"] + b["high_ue"])
         + (b["egm_per_1000"] + b["egm_per_1000:high_ue"]) * egm_grid,
         color=UE_COLORS["High unemployment"])
plt.xlabel("EGMs per 1,000 adults")
plt.ylabel("Gambling losses per adult ($/year)")
plt.show()

### The asterisk shortcut

`a * b` expands to `a + b + a:b`. Same four coefficients, shorter formula. An interaction never enters a formula without both of its parts.

In [ ]:
inter_star = ols("exp_per_adult ~ egm_per_1000 * high_ue", data=pokies).fit()

print(inter_star.summary().tables[1])

## Categories with more than two levels

`pd.qcut` cuts a variable at its quantiles, so the groups are equal sized. Three groups means cuts at the tertiles.

In [ ]:
pokies["ue_cat"] = pd.qcut(pokies["ue_pct"], 3, labels=["low", "medium", "high"])

pokies["ue_cat"].value_counts()

In [ ]:
# Where the cuts fall: the lowest and highest unemployment rate in each group
pokies.groupby("ue_cat", observed=True)["ue_pct"].agg(["min", "max", "count"]).round(2)

In [ ]:
# Three groups, three colors; the low and high colors match the two-group figures above
CAT_ORDER = ["low", "medium", "high"]
CAT_COLORS = {"low": "#5b8fc9", "medium": "#c9a24b", "high": "#c8102e"}

sns.scatterplot(data=pokies, x="egm_per_1000", y="exp_per_adult", hue="ue_cat",
                hue_order=CAT_ORDER, palette=CAT_COLORS, s=40, alpha=0.7)
plt.xlabel("EGMs per 1,000 adults")
plt.ylabel("Gambling losses per adult ($/year)")
plt.show()

### The trap: coding the levels 1, 2, 3

`.cat.codes` numbers the levels 0, 1, 2. Adding 1 gives 1, 2, 3, and the regression runs. The single coefficient forces the step from low to medium to equal the step from medium to high, which nothing in the data says.

In [ ]:
pokies["ue_code"] = pokies["ue_cat"].cat.codes + 1        # low = 1, medium = 2, high = 3

trap = ols("exp_per_adult ~ egm_per_1000 + ue_code", data=pokies).fit()

print(trap.summary().tables[1])

### `C()` makes one dummy per level

Wrap the variable in `C()` and Python builds the dummies itself, dropping one level as the reference category. Each coefficient is a gap relative to the reference, at the same pokie density; the reference category's intercept is the `Intercept` row.

In [ ]:
levels = ols("exp_per_adult ~ egm_per_1000 + C(ue_cat)", data=pokies).fit()

print(levels.summary().tables[1])

### Three parallel lines

One shared slope, three intercepts: the reference intercept, plus each dummy's coefficient.

In [ ]:
p = levels.params
intercepts = {"low": p["Intercept"],
              "medium": p["Intercept"] + p["C(ue_cat)[T.medium]"],
              "high": p["Intercept"] + p["C(ue_cat)[T.high]"]}

sns.scatterplot(data=pokies, x="egm_per_1000", y="exp_per_adult", hue="ue_cat",
                hue_order=CAT_ORDER, palette=CAT_COLORS, s=40, alpha=0.7)
for level in CAT_ORDER:
    plt.plot(egm_grid, intercepts[level] + p["egm_per_1000"] * egm_grid, color=CAT_COLORS[level])
plt.xlabel("EGMs per 1,000 adults")
plt.ylabel("Gambling losses per adult ($/year)")
plt.show()

### Changing the reference category

`Treatment(reference="high")` picks the level the others are compared with. The gaps change; the fitted lines do not.

In [ ]:
levels_high_ref = ols('exp_per_adult ~ egm_per_1000 + C(ue_cat, Treatment(reference="high"))',
                      data=pokies).fit()

levels_high_ref.params.round(1)

## Five models, one table

`summary_col` takes a list of fitted models and prints one column per model. Stars: one for 10%, two for 5%, three for 1%. A blank cell means the variable is not in that model. The standard errors and the stars are inference, which starts in week 9.

In [ ]:
table = summary_col([simple, dummy, percent, levels, inter], stars=True, float_format="%.1f",
                    model_names=["Simple", "Dummy", "Percent", "3 levels", "Interaction"],
                    info_dict={"N": lambda m: f"{int(m.nobs)}", "R-squared": lambda m: f"{m.rsquared:.2f}"})

print(table)

## Your turn

If you want to try some other things:

1. Add `SEIFA_dis_score` to the `percent` model and compare the coefficient on `egm_per_1000` with and without it
2. Interact pokie density with the three-level category, `egm_per_1000 * C(ue_cat)`, and write out the three slopes
3. Cut unemployment into four groups instead of three and compare the dummies' estimates across the two cuts

Each of these gives a different estimate, and nothing in the data picks the right specification for you. Pokies are not randomly assigned across LGAs, so none of today's numbers are causal. The Essential Concepts recordings this week are about what would make them causal.